# Заняття 10 · LIVE DEMO 2
## GOP з IDR проти intra-refresh на вузькому лінку

Справжній x264 і емулятор лінку на Python (черга з обмеженою ємністю, як на відправнику радіомодему).
Щоб уміститися в один процесор, беремо 640×360 і масштабуємо лінк: **0.8 Mbps відео на лінку 1 Mbps** — те саме навантаження 80 %, що на слайді (3.2 на 4). Пік затримки від I-кадру залежить від R / C і співвідношення I / P, а не від абсолютного бітрейту.

| Крок | Що робимо |
|---|---|
| §1 | Два енкодери з однаковим бітрейтом: GOP 30 з IDR і intra-refresh |
| §2 | Лінк 1 Mbps без втрат: модель черги зі слайда |
| §3 | Розмір кожного кадру й затримка кадру; порівняння з моделлю |
| §4 | Серія втрат 150 мс: скільки кадрів до чистої картинки — IDR, IDR на вимогу (PLI), intra-refresh |

Мітка на слайдах: «LIVE DEMO 2 · lesson10_demo2_gop_intra_refresh.ipynb» (слайд 24, розділ 03 «Енкодер і zero-copy»); GOP і модель черги — слайд 23, IDR чи intra-refresh — слайд 25, параметри енкодера — слайд 49. GStreamer живе в `streamlab.py` / `gstworker.py`, стиль графіків — у `panels.py`. Модель черги з повзунками — веб-демо 03 `lesson10_web03_gop_intra_refresh.html`.

In [ ]:
# streamlab запускає GStreamer в окремому процесі (gstworker.py), тож у ноутбуці gi не імпортується
import numpy as np
import matplotlib.pyplot as plt
import streamlab as sl
from panels import PAL, set_style

set_style()
print(sl.gst_version(), '| appsink leaky-type (1.28+):', sl.appsink_has_leaky_type())

sc = sl.Scene(640, 360, 30)
BASE = 'tune=zerolatency speed-preset=superfast bitrate=800 key-int-max=30'
ENCS = {'GOP 30, IDR': BASE, 'intra-refresh': BASE + ' intra-refresh=true'}
LINK = lambda **kw: sl.Impair(rate_mbps=1.0, keep_order=True, **kw)

### 1–2. Енкодер і лінк
Обидва варіанти — `tune=zerolatency` (без B-кадрів і lookahead), CBR 800 кбіт/с, `key-int-max=30`. В intra-refresh x264 оновлює картинку смугою intra-блоків за цикл із 30 кадрів, і окремого великого I-кадру немає.

In [ ]:
runs = {}
for name, enc in ENCS.items():
    rec, tx, link = sl.run_stream(sc, 6, enc, impair=LINK(), settle_s=0.5)
    runs[name] = (rec, tx)
    print(f'{name:14s}: кадрів {len(rec)}, втрат у лінку {link.stats["forward"]["lost"]}')

### 3. Розмір кадру і затримка кадру
Затримка кадру тут — від `push-buffer` у відправнику до кадру в appsink: кодування, черга на лінку, серіалізація, декодування.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
summary = {}
for name, col in zip(ENCS, (PAL['re'], PAL['te'])):
    rec, tx = runs[name]
    idx = sorted(i for i in tx.sizes if i >= 0)
    kb = np.array([tx.sizes[i][0] for i in idx]) / 1000
    key = np.array([tx.sizes[i][1] for i in idx])
    ax[0].bar(np.array(idx) / 30 + (0 if col == PAL['re'] else 0.012), kb, width=0.012, color=col, label=name)
    d = {r['idx']: r['age'] for r in rec if r['idx'] in tx.t_send}
    ii = np.array(sorted(d)); aa = np.array([d[i] for i in ii])
    ax[1].plot(ii / 30, aa, color=col, label=name)
    steady = aa[ii >= 30]                                    # без першої секунди розгону
    ratio = kb[key].mean() / kb[~key].mean() if key.any() and (~key).any() else float('nan')
    summary[name] = dict(p50=np.median(steady), p99=np.percentile(steady, 99), ip=ratio, kmax=kb.max())
ax[0].set_ylabel('КБ'); ax[0].set_title('Розмір кадру після x264'); ax[0].legend()
ax[1].set_ylabel('мс'); ax[1].set_xlabel('час, с'); ax[1].set_title('Затримка кадру на лінку 1 Mbps'); ax[1].legend()
plt.tight_layout(); plt.show()
for n, s in summary.items():
    print(f'{n:14s} p50 {s["p50"]:5.0f} мс  p99 {s["p99"]:5.0f} мс  «ключовий»/інші = {s["ip"]:4.1f}  найбільший кадр {s["kmax"]:.1f} КБ')
I_kbit = summary['GOP 30, IDR']['kmax'] * 8
print(f'\nмодель: найбільший I-кадр {I_kbit:.0f} кбіт / 1 Mbps = {I_kbit:.0f} мс серіалізації + кодування й декодування')

Співвідношення I / P на цій сцені помітно більше за 6 зі слайда: текстура з дрібними блоками погано стискається intra-кодуванням, а повільне панорамування добре передбачається між кадрами. На слайді 23 — ілюстративне значення; на вашій сцені його треба виміряти, як тут.

В intra-refresh x264 позначає початок кожного циклу оновлення як точку відновлення, тому прапорець «ключовий» стоїть і там, але розмір цих кадрів близький до звичайних.

### 4. Серія втрат: скільки кадрів до чистої картинки
Емулятор викидає всі пакети в інтервалі 3.40–3.55 с від старту відправника — посередині GOP. Три варіанти:

* **GOP, чекаємо наступного IDR** — нічого не просимо;
* **GOP + PLI** — після останньої втрати приймач просить IDR, запит іде RTT / 2 (RTT 80 мс), енкодер отримує `force-key-unit`;
* **intra-refresh** — нічого не просимо, картинка самовідновлюється за цикл.

Якість кадру міряємо як **частку блоків 16×16 з грубою помилкою** відносно вихідного кадру: стиснення дає дрібну рівномірну похибку, а наслідки втрат — грубі блоки. Кадри, які декодер не віддав, — це «застигла» картинка для оператора.

In [ ]:
BURST = (3.40, 3.55)
variants = {'GOP, чекаємо IDR': (BASE, None), 'GOP + PLI': (BASE, 0.08), 'intra-refresh': (ENCS['intra-refresh'], None)}
res = {}
for name, (enc, pli_rtt) in variants.items():
    rec, tx, link = sl.run_stream(sc, 6, enc, impair=LINK(blackout=[BURST]), with_psnr=True, settle_s=0.5, pli_rtt=pli_rtt)
    res[name] = ({r['idx']: r['bad'] for r in rec if r['idx'] in tx.t_send}, len(tx.t_send))
fig, ax = plt.subplots(figsize=(11, 3.8))
rows = []
for (name, (d, n)), col in zip(res.items(), (PAL['re'], PAL['or_'], PAL['te'])):
    ii = np.arange(n); bad = np.array([d.get(i, np.nan) for i in ii])
    ax.plot(ii / 30, bad, color=col, label=name, lw=2)
    after = ii[ii / 30 >= BURST[0] - 0.1]
    clean = [i for i in after if i in d and d[i] < 0.02 and all((j in d and d[j] < 0.02) for j in range(i, min(i + 5, n)))]
    first = clean[0] if clean else None
    frozen = sum(1 for i in after if first is not None and i < first and i not in d)
    rows.append((name, (first / 30 - BURST[1]) * 1000 if first is not None else float('nan'), frozen))
ax.axvspan(*BURST, color=PAL['re'], alpha=0.15)
ax.set_xlim(3.0, 5.6); ax.set_ylim(-0.03, 1.05)
ax.set_xlabel('час кадру, с'); ax.set_ylabel('частка зіпсованих блоків'); ax.set_title('Відновлення після серії втрат (розриви лінії — кадри, які декодер не віддав)')
ax.legend(); plt.tight_layout(); plt.show()
for name, ms, fr in rows:
    print(f'{name:18s} чиста картинка через {ms:5.0f} мс після кінця серії; застиглих кадрів {fr}')

### Висновки для слайда
* На лінку, завантаженому на 80 %, кожен IDR — пік затримки в сотні мілісекунд, бо великий кадр серіалізується довше за кілька кадрових періодів. Intra-refresh тримає затримку рівною.
* Ціна intra-refresh — повільне відновлення: після серії втрат картинка лікується смугою за весь цикл (тут ≈ 1.4 с). IDR на вимогу (PLI) повертає чисту картинку за RTT плюс I-кадр — але сам I-кадр знову дає пік.
* Порядок рішень на борту під РЕБ: intra-refresh для оператора + IDR на вимогу, і окремий потік з IDR для запису, де потрібен пошук.
* Співвідношення I / P залежить від сцени: міряйте його на своєму відео, перш ніж рахувати бюджет.